# Poster Labels with QR Codes

In [1]:
import pandas as pd
import qrcode
from reportlab.lib.enums import TA_JUSTIFY
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Image
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch
from reportlab.pdfgen import canvas


## Make QR Codes

For each poster associated with a paper, make a QR code to the corresponding URL.

In [ ]:
posterDF = pd.read_csv('simbuild_posters.csv')
posterDF['num'] = posterDF.index

def makeURL(paperID):
    return f'https://publications.ibpsa.org/conference/paper/?id=simbuild2026_{paperID}'

def makeQR(paperID):
    url = makeURL(paperID)
    # print(url)
    img = qrcode.make(url)
    type(img)  # qrcode.image.pil.PilImage
    qrPath = f"posters/{paperID}.png"
    img.save(qrPath)
    return qrPath

def makeLabelInfo(poster):
    info = {}
    info['num'] = poster[1]['num'] + 1
    info['presenter'] = poster[1]['presenting_author']
    info['title'] = poster[1]['title']
    if poster[1]['contribution_type'] == 'Revised Research Paper':
        info['qrPath'] = makeQR(poster[1]['paperID'])
        # print(qrPath)
    return info 

posters = []
for row in posterDF.iterrows():
    posters.append(makeLabelInfo(row))

## Make Labels

For each poster, make a label that contains the poster number (1-18), title, presenter name, and -- for posters with a paper -- QR code leading to the full paper.

In [3]:
style = getSampleStyleSheet()

def makeLabel(c, poster):
    qrSize = pheight/2
    margin = pheight/20
    
    c.setFont("Helvetica", pheight/3)
    if poster['num'] < 10:
        c.drawString(pwidth-qrSize/2-margin,2/3*pheight-margin,f"{poster['num']}")
    else:
        c.drawString(pwidth-qrSize/2-4*margin,2/3*pheight-margin,f"{poster['num']}")

    c.setFont("Helvetica", margin*2)
    c.drawString(margin,pheight-margin*3,f"{poster['presenter']}")
    c.setFont("Helvetica", margin*2)
    
    parStyle = ParagraphStyle('paragraph style',
                            fontName="Helvetica",
                            fontSize=margin,
                            parent=style['Heading2'])
    P = Paragraph(f"{poster['title']}", parStyle)
    
    # available width and height
    aW = pwidth - 3*margin - qrSize
    aH = pheight - 3*margin
    w,h = P.wrap(aW, aH)    # find required space
    if w<=aW and h<=aH:
        P.drawOn(c,margin,margin)
        aH = aH - h         # reduce the available height
    else:
        raise ValueError("Not enough room")

    if 'qrPath' in poster:
        c.drawInlineImage(f"{poster['qrPath']}", pwidth - margin - qrSize, margin, width=qrSize,height=qrSize) 


lwidth, lheight = letter  #keep for later
pwidth = lwidth/2
pheight = lheight/4

c = canvas.Canvas("posters/labels.pdf", (pwidth, pheight))
for poster in posters:
    makeLabel(c, poster)
    c.showPage()
c.save()
